In [ ]:
import os
import json
import glob
import requests
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

In [ ]:
data_dir = "../data"

input_json_dir = os.path.join(data_dir, "2_enrich", "c_attachment_information")
input_attachment_dir = os.path.join(data_dir, "1_parser", "attachments")

# Load configuration from environment variables
base_url = os.getenv("OPENWEBUI_BASE_URL")
api_key = os.getenv("OPENWEBUI_API_KEY")
openwebui_knowledge_id_attachments = os.getenv("OPENWEBUI_KNOWLEDGE_ID_ATTACHMENTS")
openwebui_knowledge_id_conversations = os.getenv("OPENWEBUI_KNOWLEDGE_ID_CONVERSATIONS")

# Validate required environment variables
if not base_url:
    raise ValueError("OPENWEBUI_BASE_URL environment variable is not set")
if not api_key:
    raise ValueError("OPENWEBUI_API_KEY environment variable is not set")
if not openwebui_knowledge_id_attachments:
    raise ValueError("OPENWEBUI_KNOWLEDGE_ID_ATTACHMENTS environment variable is not set")
if not openwebui_knowledge_id_conversations:
    raise ValueError("OPENWEBUI_KNOWLEDGE_ID_CONVERSATIONS environment variable is not set")

In [ ]:
def upload_file_to_knowledge_base(file_path, collection_id):
    """
    Upload a file to OpenWebUI and add it to a knowledge base collection.
    
    Args:
        file_path (str): Full path to the file to upload
        collection_id (str): ID of the knowledge base collection
        
    Returns:
        tuple: (success: bool, message: str)
    """
    if not os.path.exists(file_path):
        return False, f"File not found: {file_path}"
    
    filename = os.path.basename(file_path)
    upload_url = f"{base_url}/api/v1/files/"
    
    # Prepare headers for file upload
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Accept": "application/json"
    }
    # Upload the file
    with open(file_path, 'rb') as file:
        files = {'file': (filename, file)}
        response = requests.post(upload_url, headers=headers, files=files)
    
    if response.status_code != 200:
        return False, f"Upload failed - Status: {response.status_code}, Response: {response.text}"
    
    # Parse the response JSON
    response_data = response.json()
    
    # Check for errors in the response
    if 'error' in response_data and response_data['error']:
        print(f"Upload warning for {filename} - Error: {response_data['error']}")
    
    # Extract file ID
    file_id = response_data.get('id')
    if not file_id:
        raise ValueError(f"File ID not found in upload response for {filename}. Response: {response_data}")
    
    # Add file to knowledge base
    knowledge_url = f"{base_url}/api/v1/knowledge/{collection_id}/file/add"
    knowledge_headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    knowledge_payload = {"file_id": file_id}
    
    knowledge_response = requests.post(knowledge_url, headers=knowledge_headers, json=knowledge_payload)
    
    if knowledge_response.status_code == 200:
        return True, "Successfully uploaded and added to knowledge base"
    else:
        return False, f"Uploaded but failed to add to knowledge base - Status: {knowledge_response.status_code}, Response: {knowledge_response.text}"

In [ ]:
json_files = glob.glob(os.path.join(input_json_dir, "*.json"))
for idx, file in enumerate(json_files, 1):
    if idx < 863:
        continue
    filename = os.path.basename(file)
    with open(file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # Get all attachment filenames from the JSON data (nested in Messages -> Attachments)
    attachments_to_upload = []
    if 'Messages' in data:
        for message in data['Messages']:
            if 'Attachments' in message:
                for attachment in message['Attachments']:
                    if 'path' in attachment:
                        attachments_to_upload.append(attachment['path'])
        
    # Upload each attachment
    for attachment_idx, attachment_filename in enumerate(attachments_to_upload, 1):
        if attachment_filename.lower().endswith(('.zip', '.mov', '.opus')):
            print(f"({idx}/{len(json_files)}) Attachment {attachment_idx}: Skipped file - {attachment_filename}")
            continue
            
        attachment_path = os.path.join(input_attachment_dir, attachment_filename)
        success, message = upload_file_to_knowledge_base(attachment_path, openwebui_knowledge_id_attachments)
        
        if success:
            print(f"({idx}/{len(json_files)}) Attachment {attachment_idx}: {message} - {attachment_filename}")
        else:
            print(f"({idx}/{len(json_files)}) Attachment {attachment_idx}: {message} - {attachment_filename}")
    
    # Upload the JSON conversation file to conversations collection
    success, message = upload_file_to_knowledge_base(file, openwebui_knowledge_id_conversations)
    
    if success:
        print(f"({idx}/{len(json_files)}) Conversation: {message} - {filename}")
    else:
        print(f"({idx}/{len(json_files)}) Conversation: {message} - {filename}")